In [20]:
import pandas as pd

In [21]:
pd.__version__

'2.2.3'

In [22]:
# tem_df = pd.read_excel('online_retail_II.xlsx')
# this will only read the first sheet but our file has 2 sheets.

# sheets = pd.read_excel('online_retail_II.xlsx',sheet_name=['Year 2009-2010','Year 2010-2011'])
sheets = pd.read_excel('online_retail_II.xlsx',sheet_name=None)  # reads all sheets

# this returns a dict of {sheet_name: DataFrame}
# if sheets already have an index column with no specific purpose then remove it and let the index of pd stay.

In [23]:
print(sheets.keys())

dict_keys(['Year 2009-2010', 'Year 2010-2011'])


In [24]:
# to check if the there is any overlap

for name, d in sheets.items():
    print(name, d["InvoiceDate"].min(), "->", d["InvoiceDate"].max())

# first, second = sheets.values()
# print("Rows identical in both sheets:", len(first.merge(second, how="inner")))

Year 2009-2010 2009-12-01 07:45:00 -> 2010-12-09 20:01:00
Year 2010-2011 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [25]:
# checking if overlaps length are same

a, b = sheets["Year 2009-2010"], sheets["Year 2010-2011"]

a_ov = a[a["InvoiceDate"] >= b["InvoiceDate"].min()]   # sheet 1 rows inside the shared window
b_ov = b[b["InvoiceDate"] <= a["InvoiceDate"].max()]   # sheet 2 rows inside the shared window
print(len(a_ov), len(b_ov))

22523 22523


In [26]:
# Is the overlap window an exact copy in both sheets?
a, b = sheets["Year 2009-2010"], sheets["Year 2010-2011"]
cut = a["InvoiceDate"].max()                    # last timestamp in sheet 1

a_ov = a[a["InvoiceDate"] >= b["InvoiceDate"].min()]   # sheet 1 rows inside the shared window
b_ov = b[b["InvoiceDate"] <= cut]                       # sheet 2 rows inside the shared window

def as_sorted_text(d):
    # Invoice and StockCode mix numbers and text, so compare text versions of every value.
    # Sorting puts both copies in the same order; reset_index drops the old row labels.
    t = d.astype(str)
    return t.sort_values(list(t.columns)).reset_index(drop=True)

identical = as_sorted_text(a_ov).equals(as_sorted_text(b_ov))
print("rows in window:", len(a_ov), len(b_ov), "| identical:", identical)

# df = pd.concat(sheets.values(), ignore_index=True)

# use this concat with caution as any mismatch in column name or extra columns are simply merged.
# ignore_index=True helps to ignore individual index, sheets are merged and then indexed

rows in window: 22523 22523 | identical: True


In [27]:
assert identical, "Overlap rows differ: inspect them before removing anything"

b = b[b["InvoiceDate"] > cut]                   # keep only the rows that are new in sheet 2
df = pd.concat([a, b], ignore_index=True)
print(df.shape)

(1044848, 8)


In [28]:
print(df.head())


print('\n',"----Column Names----",'\n')
# get the columns
print(df.columns)


  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  

 ----Column Names---- 

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')


In [29]:
# rename the columns to remove spaces
df = df.rename(columns={"Invoice":"invoice","StockCode":"stock_code","Description":"description","Quantity":"quantity","InvoiceDate":"invoice_date","Price":"price","Customer ID":"customer_id","Country":"country"})

# do not forget to reassign this rename to df as it will give a new df
# df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [30]:
print("----Shape of dataframe----",'\n')
print(df.shape,'\n')

print("----Info of dataframe----",'\n')
print(df.info(),'\n')

print("----Null values for each column----",'\n')
print(df.isna().sum())

----Shape of dataframe---- 

(1044848, 8) 

----Info of dataframe---- 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1044848 non-null  object        
 1   stock_code    1044848 non-null  object        
 2   description   1040573 non-null  object        
 3   quantity      1044848 non-null  int64         
 4   invoice_date  1044848 non-null  datetime64[ns]
 5   price         1044848 non-null  float64       
 6   customer_id   809561 non-null   float64       
 7   country       1044848 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 63.8+ MB
None 

----Null values for each column---- 

invoice              0
stock_code           0
description       4275
quantity             0
invoice_date         0
price                0
customer_id     235287
country 

In [31]:
# df.to_parquet('online_retail_II_parquet', index=False)

''' this line fails with error
  ArrowInvalid: ("Could not convert 'C489449' with type str: tried to convert to int64",
  Conversion failed for column Invoice with type object')
'''
# before conversion a column data must be of unique datatype, few columns are read as object datatype as some entries are in 'str' and some entries are in 'int' datatype.

' this line fails with error \n  ArrowInvalid: ("Could not convert \'C489449\' with type str: tried to convert to int64",\n  Conversion failed for column Invoice with type object\')\n'

In [32]:
for col in df.columns.tolist():
    print(f"--- {col} ---")
    print(df[col].map(type).value_counts())
    print('nan count for this column : ',df[col].isna().sum())
    print()

--- invoice ---
invoice
<class 'int'>    1025677
<class 'str'>      19171
Name: count, dtype: int64
nan count for this column :  0

--- stock_code ---
stock_code
<class 'int'>    912611
<class 'str'>    132237
Name: count, dtype: int64
nan count for this column :  0

--- description ---
description
<class 'str'>      1040569
<class 'float'>       4275
<class 'int'>            4
Name: count, dtype: int64
nan count for this column :  4275

--- quantity ---
quantity
<class 'int'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- invoice_date ---
invoice_date
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- price ---
price
<class 'float'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- customer_id ---
customer_id
<class 'float'>    1044848
Name: count, dtype: int64
nan count for this column :  235287

--- country ---
country
<class 'str'>    1044848
Name: count, dtype:

In [33]:
'''
for col in ["invoice", "stock_code"]:
    s = df[col].astype(str)
    digits_only = s.str.fullmatch(r"\d+")          # True if the value is purely digits
    print(f"--- {col} ---")
    print("all-digit share:", digits_only.mean())
    print("non-digit count:", (~digits_only).sum())
    print(s[~digits_only].unique()[:20])           # sample of the offenders
    print(s[~digits_only].str[0].value_counts().head(10))   # which letters they start with
    print()
'''

# run this in case ever want to verify if the str are just 'int' as in '20753' rather than containing any character type data.

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_544/613418174.py:4: SyntaxWarning: invalid escape sequence '\d'
  digits_only = s.str.fullmatch(r"\d+")          # True if the value is purely digits


'\nfor col in ["invoice", "stock_code"]:\n    s = df[col].astype(str)\n    digits_only = s.str.fullmatch(r"\\d+")          # True if the value is purely digits\n    print(f"--- {col} ---")\n    print("all-digit share:", digits_only.mean())\n    print("non-digit count:", (~digits_only).sum())\n    print(s[~digits_only].unique()[:20])           # sample of the offenders\n    print(s[~digits_only].str[0].value_counts().head(10))   # which letters they start with\n    print()\n'

In [34]:
# no nulls -> plain astype(str) is safe
df["invoice"]   = df["invoice"].astype(str)
df["stock_code"] = df["stock_code"].astype(str)

# has nulls -> use the nullable "string" dtype so NaN stays missing (<NA>) instead of becoming the text "nan"
df["description"] = df["description"].astype("string")

# float with NaN -> nullable integer (capital I)
df["customer_id"] = df["customer_id"].astype("Int64")

# df["country"] = df["country"].astype("string")   # optional, for consistency

In [35]:
for col in df.columns.tolist():
    print(f"--- {col} ---")
    print(df[col].map(type).value_counts())
    print('nan count for this column : ',df[col].isna().sum())
    print()

--- invoice ---
invoice
<class 'str'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- stock_code ---
stock_code
<class 'str'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- description ---
description
<class 'str'>                            1040573
<class 'pandas._libs.missing.NAType'>       4275
Name: count, dtype: int64
nan count for this column :  4275

--- quantity ---
quantity
<class 'int'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- invoice_date ---
invoice_date
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- price ---
price
<class 'float'>    1044848
Name: count, dtype: int64
nan count for this column :  0

--- customer_id ---
customer_id
<class 'float'>    1044848
Name: count, dtype: int64
nan count for this column :  235287

--- country ---
country
<class 'str'>    1044848
Name: count, dtype: int64
nan count for this colum

In [36]:
df.to_parquet("online_retail_II_clean.parquet", index=False)

In [37]:
df.to_excel("online_retail_II_clean.xlsx", index=False)

# An Excel sheet holds at most 1,048,576 rows, and The Excel save will fail as a single sheet if your df has more rows. to_excel raises a "sheet is too large" error. If you do want Excel, write the data as two sheets:

# with pd.ExcelWriter("online_retail_II_clean.xlsx") as writer:
#     df.iloc[:500_000].to_excel(writer, sheet_name="part1", index=False)
#     df.iloc[500_000:].to_excel(writer, sheet_name="part2", index=False)